# TUẦN 41 — TÍCH HỢP ATTENTION-FUSION VÀO PIPELINE UNIFASHION


## 1. Đồng bộ repo và kiểm tra GPU

In [2]:
from pathlib import Path
import subprocess
import sys

import torch

# Repo khóa luận đã có trên máy local
PROJECT = Path(
    r"C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep"
).resolve()

# Repo UniFashion upstream nằm cạnh repo khóa luận
UPSTREAM = PROJECT.parent / "UniFashion-upstream"

REPO_URL = "https://github.com/okok0814/khoaluantotnghiep.git"
UPSTREAM_URL = "https://github.com/xiangyu-mm/UniFashion.git"


def run(*cmd, cwd=None, check=True):
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd))
    return subprocess.run(
        cmd,
        cwd=cwd,
        check=check,
        text=True
    )


# 1. Kiểm tra repo khóa luận, KHÔNG clone lại
if not PROJECT.is_dir():
    raise FileNotFoundError(
        f"Không tìm thấy repo khóa luận: {PROJECT}\n"
        "Hãy sửa PROJECT thành đường dẫn repo thực tế trên máy."
    )

if not (PROJECT / ".git").exists():
    raise RuntimeError(f"Thư mục không phải Git repo: {PROJECT}")

result = subprocess.run(
    ["git", "remote", "get-url", "origin"],
    cwd=PROJECT,
    capture_output=True,
    text=True
)

if result.returncode != 0:
    raise RuntimeError("Repo khóa luận chưa cấu hình remote origin.")

actual_url = result.stdout.strip().removesuffix(".git")
expected_url = REPO_URL.removesuffix(".git")

if actual_url.lower() != expected_url.lower():
    raise RuntimeError(
        f"Remote repo không đúng.\n"
        f"Đang có: {result.stdout.strip()}\n"
        f"Mong đợi: {REPO_URL}"
    )

print("PROJECT:", PROJECT)
print("Remote:", result.stdout.strip())

# 2. Kiểm tra branch và thay đổi hiện tại, không tự sửa Git
run("git", "branch", "--show-current", cwd=PROJECT)
run("git", "status", "--short", cwd=PROJECT)

# 3. Chỉ clone repo upstream nếu chưa có
if not UPSTREAM.exists():
    print("\nChưa có upstream, tiến hành clone mã nguồn UniFashion...")
    run("git", "clone", UPSTREAM_URL, UPSTREAM)
else:
    if not (UPSTREAM / ".git").exists():
        raise RuntimeError(
            f"Thư mục upstream đã tồn tại nhưng không phải Git repo: {UPSTREAM}"
        )
    print("\nĐã có upstream, không clone lại.")

# 4. In thông tin hai repo
print("\nPROJECT HEAD:")
run("git", "log", "-1", "--oneline", cwd=PROJECT)

print("\nUPSTREAM HEAD:")
run("git", "log", "-1", "--oneline", cwd=UPSTREAM)

# 5. Kiểm tra môi trường
print("\nPython:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Đang dùng CPU; cần cân nhắc thời gian fine-tune.")


PROJECT: C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep
Remote: https://github.com/okok0814/khoaluantotnghiep.git
+ git branch --show-current
+ git status --short

Chưa có upstream, tiến hành clone mã nguồn UniFashion...
+ git clone https://github.com/xiangyu-mm/UniFashion.git C:\Users\Nguyen Ho Vinh  Hien\UniFashion-upstream

PROJECT HEAD:
+ git log -1 --oneline

UPSTREAM HEAD:
+ git log -1 --oneline

Python: 3.11.9
PyTorch: 2.13.0+cpu
CUDA available: False
Đang dùng CPU; cần cân nhắc thời gian fine-tune.


## 2. Kiểm tra file và chạy smoke test (không train)

In [3]:
required = [
    PROJECT / "src/unifashion_runtime.py",
    PROJECT / "src/attention_fusion.py",
    PROJECT / "src/fashioniq_unifashion_dataset.py",
    PROJECT / "scripts/test_attention_fusion_adapter.py",
    PROJECT / "scripts/train_unifashion_finetune.py",
    PROJECT / "scripts/evaluate_unifashion_fashioniq.py",
    UPSTREAM / "src/lavis/models/blip2_models/blip2_qformer_cir_rerank.py",
]
missing = [p for p in required if not p.is_file()]
for p in required: print(("OK   " if p.is_file() else "MISS "), p)
if missing: raise FileNotFoundError("Thiếu file code bắt buộc. Dừng trước khi train.")

os.chdir(PROJECT)
result = subprocess.run([sys.executable, "scripts/test_attention_fusion_adapter.py"], cwd=PROJECT, text=True, capture_output=True)
print(result.stdout)
if result.stderr: print(result.stderr[-4000:])
if result.returncode != 0: raise RuntimeError("Smoke test thất bại; chưa được chạy train.")
for marker in ["GATE_256_FORWARD_BACKWARD=PASS", "UPSTREAM_TRAIN_AND_INFERENCE_INJECTION=PASS", "ATTENTION_ADAPTER_SMOKE_TEST=PASS"]:
    if marker not in result.stdout: raise RuntimeError(f"Thiếu marker: {marker}")
print("SMOKE TEST: PASS")

OK    C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep\src\unifashion_runtime.py
OK    C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep\src\attention_fusion.py
OK    C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep\src\fashioniq_unifashion_dataset.py
OK    C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep\scripts\test_attention_fusion_adapter.py
OK    C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep\scripts\train_unifashion_finetune.py
OK    C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep\scripts\evaluate_unifashion_fashioniq.py
OK    C:\Users\Nguyen Ho Vinh  Hien\UniFashion-upstream\src\lavis\models\blip2_models\blip2_qformer_cir_rerank.py
GATE_256_FORWARD_BACKWARD=PASS
UPSTREAM_TRAIN_AND_INFERENCE_INJECTION=PASS
ATTENTION_ADAPTER_SMOKE_TEST=PASS

SMOKE TEST: PASS


## 3. Đặt đường dẫn dữ liệu và checkpoint

Cell dưới đây kiểm tra các asset tại đường dẫn mặc định. Nếu đã upload ở chỗ khác, sửa biến trong cell này. Không tự tải các file nhiều GB.

In [5]:
from pathlib import Path

PROJECT = Path(
    r"C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep"
).resolve()

# Các thư mục có thể chứa dữ liệu FashionIQ trên máy local
DATA_ROOTS = [
    PROJECT / "data",
    PROJECT / "datasets",
    PROJECT / "unifashion_sanity" / "data",
    PROJECT.parent / "unifashion_sanity" / "data",
]

IMAGE_ROOTS = [
    PROJECT / "images",
    PROJECT / "data" / "images",
    PROJECT / "datasets" / "images",
    PROJECT / "unifashion_sanity" / "images",
    PROJECT.parent / "unifashion_sanity" / "images",
]


def find_first_existing(candidates):
    return next((p for p in candidates if p.exists()), None)


train_csv = find_first_existing([
    root / "train.csv" for root in DATA_ROOTS
])
val_csv = find_first_existing([
    root / "val.csv" for root in DATA_ROOTS
])
images_dir = find_first_existing(IMAGE_ROOTS)

checks = [
    ("Train CSV", train_csv),
    ("Validation CSV", val_csv),
    ("Images", images_dir),
]

for label, path in checks:
    if path is not None:
        print(f"OK — {label}: {path}")
    else:
        print(f"MISSING — {label}")

missing = [label for label, path in checks if path is None]

if missing:
    print("\nChưa tìm thấy:", ", ".join(missing))
    print("\nCác thư mục dữ liệu hiện có trong repo:")
    for folder in [PROJECT / "data", PROJECT / "datasets"]:
        if folder.exists():
            print(f"\n{folder}")
            for item in list(folder.iterdir())[:30]:
                print(" -", item.name)

    raise FileNotFoundError(
        "Chưa tìm thấy đủ dữ liệu train/validation/ảnh. "
        "Cần xác định vị trí dữ liệu thực tế trước khi train."
    )

print("\nĐã tìm thấy các đường dẫn dữ liệu.")

MISSING — Train CSV
MISSING — Validation CSV
MISSING — Images

Chưa tìm thấy: Train CSV, Validation CSV, Images

Các thư mục dữ liệu hiện có trong repo:

C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep\data
 - processed


FileNotFoundError: Chưa tìm thấy đủ dữ liệu train/validation/ảnh. Cần xác định vị trí dữ liệu thực tế trước khi train.

In [7]:

from pathlib import Path
import os

PROJECT = Path(
    r"C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep"
).resolve()

SKIP_DIRS = {
    ".git", ".venv", "venv", "__pycache__",
    "node_modules", "appdata", "docker"
}

csv_hits = []
image_dirs = []
image_files = []
image_exts = {".jpg", ".jpeg", ".png", ".webp"}

def safe_walk_error(error):
    # Bỏ qua thư mục hoặc file Windows không cho truy cập
    print(f"Bỏ qua đường dẫn không truy cập được: {error.filename}")

print("Đang tìm dataset bên trong dự án:", PROJECT)

for current, dirs, files in os.walk(
    PROJECT,
    topdown=True,
    onerror=safe_walk_error
):
    # Không đi vào các thư mục không cần thiết
    dirs[:] = [
        d for d in dirs
        if d.lower() not in SKIP_DIRS
    ]

    current_path = Path(current)

    for filename in files:
        p = current_path / filename

        if p.suffix.lower() == ".csv":
            csv_hits.append(p)

        if p.suffix.lower() in image_exts:
            image_files.append(p)

    for dirname in dirs:
        if any(
            word in dirname.lower()
            for word in ("image", "fashioniq", "fashion_iq", "img")
        ):
            image_dirs.append(current_path / dirname)

print("\nFILE CSV TÌM THẤY")
if csv_hits:
    for p in csv_hits[:100]:
        print(p)
else:
    print("Không tìm thấy file CSV trong dự án.")

print("\nTHƯ MỤC CÓ TÊN LIÊN QUAN DATASET/ẢNH")
if image_dirs:
    for p in image_dirs[:100]:
        print(p)
else:
    print("Không tìm thấy thư mục có tên liên quan.")

print("\nẢNH TÌM THẤY")
print(f"Tổng số ảnh tìm thấy: {len(image_files)}")
for p in image_files[:20]:
    print(p)

Đang tìm dataset bên trong dự án: C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep

FILE CSV TÌM THẤY
C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep\data\processed\fashioniq_triplets_test.csv
C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep\data\processed\fashioniq_triplets_train.csv
C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep\data\processed\fashioniq_triplets_val.csv
C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep\data\processed\week40_attention_fusion\attention_vs_additive_recall.csv
C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep\data\processed\week40_attention_fusion\training_history.csv
C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep\logs\week9_fashioniq_evaluation\comparison.csv
C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep\logs\week9_fashioniq_evaluation\error_review_examples.csv
C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep\logs\week9_fashioniq_evaluation\query_transitions.csv
C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep\logs\week9_fashioniq_evaluation\baselines\fashi

## 4. Xác nhận tham số train/evaluate

In [9]:
from pathlib import Path
import sys
import subprocess

PROJECT = Path(
    r"C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep"
).resolve()

print("PROJECT:", PROJECT)
print("PROJECT tồn tại:", PROJECT.is_dir())
print("Python executable:", sys.executable)
print("Python executable tồn tại:", Path(sys.executable).is_file())

print("\nKIỂM TRA CẤU TRÚC DỰ ÁN")
for rel in [
    "scripts",
    "scripts/train_unifashion_finetune.py",
    "scripts/evaluate_unifashion_fashioniq.py",
    ".git",
]:
    p = PROJECT / rel
    print(f"{'OK' if p.exists() else 'MISSING'}: {p}")

print("\nCÁC SCRIPT HIỆN CÓ")
scripts_dir = PROJECT / "scripts"
if scripts_dir.is_dir():
    for p in sorted(scripts_dir.glob("*.py")):
        print(p.name)
else:
    print("Không tìm thấy thư mục scripts trong PROJECT.")

print("\nKIỂM TRA PYTHON")
try:
    result = subprocess.run(
        [sys.executable, "--version"],
        cwd=str(PROJECT) if PROJECT.is_dir() else None,
        capture_output=True,
        text=True,
        check=False,
    )
    print("Return code:", result.returncode)
    print(result.stdout.strip() or result.stderr.strip())
except OSError as e:
    print(type(e).__name__ + ":", e)

PROJECT: C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep
PROJECT tồn tại: True
Python executable: C:\Users\Nguyen Ho Vinh  Hien\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe
Python executable tồn tại: True

KIỂM TRA CẤU TRÚC DỰ ÁN
OK: C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep\scripts
OK: C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep\scripts\train_unifashion_finetune.py
OK: C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep\scripts\evaluate_unifashion_fashioniq.py
OK: C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep\.git

CÁC SCRIPT HIỆN CÓ
baseline_retrieval.py
benchmark_unifashion_t4.py
check_unifashion_environment.py
download_unifashion.py
evaluate_baseline_fashioniq.py
evaluate_unifashion_fashioniq.py
extract_image_embeddings_resnet_clip.py
extract_unifashion_sanity_evidence.py
fashioniq_integrity_check.py
package_fashioniq_evaluation.py
package_unifashion_finetune.py
package_unifashion_sanity.py
prepare_fashioniq_evaluation.py

## 5. Fine-tune — bước có dùng GPU

Chỉ điền tham số sau khi đọc output `--help` và xác nhận cấu hình dataset/checkpoint. Mặc định cố ý không chạy để tránh tiêu tốn GPU do tham số chưa xác minh.

In [11]:
from pathlib import Path
import sys
import subprocess
import torch

PROJECT = Path(
    r"C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep"
).resolve()

train_script = PROJECT / "scripts" / "train_unifashion_finetune.py"

print("PROJECT:", PROJECT)
print("Thư mục dự án tồn tại:", PROJECT.is_dir())
print("Script train tồn tại:", train_script.is_file())
print("Python:", sys.executable)
print("CUDA khả dụng:", torch.cuda.is_available())

if not PROJECT.is_dir():
    raise FileNotFoundError(f"Không tìm thấy dự án: {PROJECT}")

if not train_script.is_file():
    print("\nCHƯA THỂ TRAIN: Không tìm thấy script train tại đường dẫn trên.")
    print("Kiểm tra thư mục scripts và tên file thực tế trước.")
else:
    result = subprocess.run(
        [sys.executable, str(train_script), "--help"],
        cwd=str(PROJECT),
        capture_output=True,
        text=True,
        check=False,
    )

    print("\nTRAIN SCRIPT HELP")
    print(result.stdout if result.stdout else result.stderr)

    if result.returncode != 0:
        print("\nCHƯA THỂ TRAIN: Script --help chạy không thành công.")
    else:
        print("\nĐã lấy được tham số của script. Chưa chạy train.")

PROJECT: C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep
Thư mục dự án tồn tại: True
Script train tồn tại: True
Python: C:\Users\Nguyen Ho Vinh  Hien\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe
CUDA khả dụng: False

TRAIN SCRIPT HELP
usage: train_unifashion_finetune.py [-h] [--config CONFIG]
                                    [--upstream UPSTREAM] [--assets ASSETS]
                                    --data-dir DATA_DIR --images-dir
                                    IMAGES_DIR --output OUTPUT
                                    [--resume RESUME]
                                    [--stop-after-epoch STOP_AFTER_EPOCH]

Full FashionIQ training with the upstream optimizer/schedule and resumable
runs.

options:
  -h, --help            show this help message and exit
  --config CONFIG
  --upstream UPSTREAM
  --assets ASSETS
  --data-dir DATA_DIR
  --images-dir IMAGES_DIR
  --output OUTPUT
  --resume RESUME
  --stop-after-epoch STOP_A

In [13]:
from pathlib import Path
import torch

PROJECT = Path(
    r"C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep"
).resolve()

print("KIỂM TRA GPU")
print("PyTorch version:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Chưa có CUDA GPU khả dụng trong môi trường Python hiện tại.")

print("\nCẤU HÌNH VÀ DATASET")
for pattern in [
    "*.yaml", "*.yml", "*.json",
    "data/**/*", "configs/**/*",
]:
    print(f"\nTìm theo: {pattern}")
    matches = list(PROJECT.glob(pattern))
    for p in matches[:30]:
        print(p.relative_to(PROJECT))
    if not matches:
        print("Không tìm thấy.")

KIỂM TRA GPU
PyTorch version: 2.13.0+cpu
CUDA build: None
CUDA available: False
Chưa có CUDA GPU khả dụng trong môi trường Python hiện tại.

CẤU HÌNH VÀ DATASET

Tìm theo: *.yaml
Không tìm thấy.

Tìm theo: *.yml
Không tìm thấy.

Tìm theo: *.json
Không tìm thấy.

Tìm theo: data/**/*
data\processed
data\processed\fashioniq_triplets_test.csv
data\processed\fashioniq_triplets_train.csv
data\processed\fashioniq_triplets_val.csv
data\processed\week40_attention_fusion
data\processed\week40_attention_fusion\attention_gated_fusion.pt
data\processed\week40_attention_fusion\attention_vs_additive_recall.csv
data\processed\week40_attention_fusion\training_history.csv

Tìm theo: configs/**/*
configs\unifashion_finetune_colab.json
configs\unifashion_sanity.json


## 6. Đánh giá Recall

Sau khi train xong, đánh giá bằng snapshot checkpoint đã lưu riêng; không đánh giá `last.pt` đang được cập nhật. Sanity subset chỉ kiểm tra luồng chạy, không thay thế benchmark FashionIQ đầy đủ.

In [12]:
CONFIRMED_EVAL_ARGS = None  # thay bằng tham số chính xác theo --help
if not CONFIRMED_EVAL_ARGS:
    print("Chưa chạy evaluation: điền CONFIRMED_EVAL_ARGS sau khi có snapshot checkpoint.")
else:
    run([sys.executable, "scripts/evaluate_unifashion_fashioniq.py", *CONFIRMED_EVAL_ARGS], cwd=PROJECT)

Chưa chạy evaluation: điền CONFIRMED_EVAL_ARGS sau khi có snapshot checkpoint.


## 7. Ghi kết quả

Ghi lại commit, checkpoint và SHA-256, cấu hình, seed, số bước, Recall@k, và ghi rõ đây là sanity subset hay validation đầy đủ. Không báo Recall nếu evaluation chưa hoàn tất và chưa sinh báo cáo.